# Proyek Akhir: Menyelesaikan Permasalahan Perusahaan Edutech

- Nama: I Made Sandika Wijaya
- Email: sandikakadek2018@gmail.com
- Id Dicoding: I Made Sandika Wijaya

## Persiapan

### Menyiapkan library yang dibutuhkan

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_curve

from imblearn.over_sampling import SMOTE
import joblib

### Menyiapkan data yang akan digunakan

In [ ]:
employee_df = pd.read_csv(
    "https://raw.githubusercontent.com/dicodingacademy/dicoding_dataset/main/employee/employee_data.csv",
    encoding='windows-1252'
)
employee_df.head(5)

## Data Understanding

In [ ]:
print("Shape:", employee_df.shape)
print("\nInfo:")
print(employee_df.info())

print("\nDescribe:")
print(employee_df.describe())

print("\nMissing Values:")
print(employee_df.isnull().sum())

print("\nDuplicated Rows:")
print(employee_df.duplicated().sum())

print("\nKolom konstan:")
for col in employee_df.columns:
    if employee_df[col].nunique() == 1:
        print(col)

In [ ]:
employee_df.head()

### Data Cleaning

In [ ]:
print("\nShape sebelum cleaning:", employee_df.shape)

# Menghapus baris dengan nilai yang hilang
employee_df.dropna(inplace=True)

# Menghapus kolom konstan dan kolom yang tidak relevan
drop_cols = ['EmployeeId','EmployeeCount','Over18','StandardHours']
employee_df.drop(columns=[col for col in drop_cols if col in employee_df.columns], inplace=True)

print("\nShape setelah cleaning:", employee_df.shape)

## Exploratory Data Analysis

In [ ]:
num_df = employee_df.select_dtypes(include='number')
cat_df = employee_df.select_dtypes(exclude='number')

print("Numerical Columns Shape:", num_df.shape)
print("Numerical Columns:", num_df.columns.tolist())

print("\nCategorical Columns Shape:", cat_df.shape)
print("Categorical Columns:", cat_df.columns.tolist())

In [ ]:
num_df.hist(figsize=(20,10))
plt.suptitle("Distribusi Variabel Numerik")
plt.show()

In [ ]:
for col in cat_df.columns:
    plt.figure(figsize=(6,4))
    sns.countplot(x=col, data=employee_df)
    plt.title(f"Distribusi {col}")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
# Categorical vs Target
for col in cat_df.columns:
    plt.figure(figsize=(6,4))
    sns.countplot(x=col, hue='Attrition', data=employee_df)
    plt.title(f"{col} vs Attrition")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
# Numerical vs Target
for col in num_df.columns:
    if col != 'Attrition':
        plt.figure(figsize=(6,4))
        sns.boxplot(x='Attrition', y=col, data=employee_df)
        plt.title(f"{col} vs Attrition")
        plt.show()

In [ ]:
for col in num_df.columns:
    plt.figure(figsize=(6,3))
    sns.boxplot(x=employee_df[col])
    plt.title(f"Outlier Detection: {col}")
    plt.show()

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(num_df.corr(), cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
print("\nDistribusi Attrition:")
print(employee_df['Attrition'].value_counts())
print(employee_df['Attrition'].value_counts(normalize=True))

sns.countplot(x='Attrition', data=employee_df)
plt.title("Distribusi Attrition")
plt.show()

## Data Preparation / Preprocessing

In [ ]:
le = LabelEncoder()
employee_df['Attrition'] = le.fit_transform(employee_df['Attrition'])

In [ ]:
employee_df = pd.get_dummies(employee_df, drop_first=True)

encoded_columns = employee_df.drop("Attrition", axis=1).columns
joblib.dump(encoded_columns, "model/encoded_columns.pkl")

In [ ]:
X = employee_df.drop('Attrition', axis=1)
y = employee_df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:")
print(pd.Series(y_train).value_counts())

print("\nAfter SMOTE:")
print(pd.Series(y_train_res).value_counts())

## Modeling

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train_res, y_train_res)

## Evaluation

In [ ]:
# Probabilitas + threshold tuning
y_prob = rf.predict_proba(X_test)[:,1]
y_pred = (y_prob > 0.35).astype(int)

print("=== Base Model ===")
print(classification_report(y_test, y_pred))

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label='ROC Curve')
plt.plot([0,1], [0,1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

In [ ]:
feat_imp = pd.Series(rf.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=False)

print("Top Features:")
print(feat_imp.head(45))

## Save Model

In [ ]:
joblib.dump(rf, "model/rf_model_attrition.pkl")
joblib.dump(scaler, "model/scaler.pkl")